# Modern RL: Preferences, World Models, and Machine Minds

Companion notebook for Tutorial 3, Chapter 25. Runs every code block from the chapter.

Chapter: https://josephausterweil.github.io/probintro/intro2/25_modern_rl_world_models/

In [ ]:
# Colab setup
!pip -q install "genjax==0.10.3" "jax==0.5.3" 2>/dev/null

In [ ]:
import jax, jax.numpy as jnp
from jax import random, vmap
from genjax import gen, normal, flip, ChoiceMap

K = 3                                              # three candidate answers
ITEMS = ["A", "B", "C"]
TRUE_R = jnp.array([2.0, 0.5, -1.0])               # hidden quality (A > B > C)
PAIRS = jnp.array([[0, 1], [0, 2], [1, 2]])        # the three head-to-head comparisons

@gen
def pref_model(pairs):                             # Bradley-Terry: P(i > j) = sigmoid(r_i - r_j)
    r = jnp.array([normal(0.0, 2.0) @ f"r_{k}" for k in range(K)])     # latent reward, normal prior
    for n in range(pairs.shape[0]):
        i, j = pairs[n, 0], pairs[n, 1]
        flip(jax.nn.sigmoid(r[i] - r[j])) @ f"pref_{n}"                 # True => i preferred over j
    return r

def make_dataset(n_each=30, key=random.PRNGKey(0)):                     # noisy human prefs from TRUE_R
    pairs = jnp.repeat(PAIRS, n_each, axis=0)
    pi = jax.nn.sigmoid(TRUE_R[pairs[:, 0]] - TRUE_R[pairs[:, 1]])
    return pairs, random.bernoulli(key, pi)

def recover_reward(pairs, prefs, key=random.PRNGKey(1), n_particles=40000):
    cm = ChoiceMap.d({f"pref_{n}": bool(prefs[n]) for n in range(prefs.shape[0])})
    def one(k):
        tr, w = pref_model.importance(k, cm, (pairs,))                  # condition on the preferences
        ch = tr.get_choices()
        return jnp.array([ch[f"r_{i}"] for i in range(K)]), w
    rs, ws = vmap(one)(random.split(key, n_particles))                 # sampling-importance-resampling
    return (jax.nn.softmax(ws)[:, None] * rs).sum(0)                    # posterior-mean reward

def center(r):                                     # remove the unidentifiable additive shift
    return r - jnp.mean(r)

In [ ]:
pairs, prefs = make_dataset()
r_hat = recover_reward(pairs, prefs)
print(f"{prefs.shape[0]} human preferences from a hidden quality A=2.0, B=0.5, C=-1.0 (A>B>C).")
print("recovered reward from the preferences alone (mean-centered):")
print("  item   true   recovered")
for k in range(K):
    print(f"   {ITEMS[k]}    {float(center(TRUE_R)[k]):+5.2f}    {float(center(r_hat)[k]):+5.2f}")
order = [ITEMS[i] for i in jnp.argsort(-r_hat)]
print(f"recovered ranking: {' > '.join(order)}")